# Multi-Source Visual Search Engine (MSSearch) Diagnostic & Visualization Suite
## End-to-End Multi-Image Pipeline Instrumenting Stages 1–6 with Continuous Enclosure Sub-Canvas Aggregation

This diagnostic notebook demonstrates, executes, and visualizes **MSSearch** on a multi-source visual search environment:
- **Candidate Pool $\mathcal{C} = \{I_1, I_2, \dots, I_N\}$**: The ground-truth high-resolution target image is placed alongside $N-1$ distractors sampled within a local index window ($\pm 5$) from the dataset.
- **Unified Stages 3, 4, and 5 Implementation (`draft/improve_all`)**:
  - **Stage 3 Partitioning**: Quadtree decomposition tracks `parent_id` and `level` lineage, computing structural entropy $H(p)$ and SigLIP alignment $S(p)$ with marginal gain termination.
  - **Stage 4 Grouping & Gating**: Purges entropy from ranking; sorts confirmed detection members by maximum detected bounding box area ($\max w \times h$) descending; gates recursive repartitioned children via SAM3 re-detection with continuous unboosted scores in $[0.0, 1.0]$; applies dynamic retention elasticity at cutoff.
  - **Stage 5 Continuous Enclosure Sub-Canvas Aggregation**: Re-fuses seam-adjacent quadtree siblings into atomic solid boxes; greedily partitions clusters into continuous photographic sub-canvases bounded strictly to $\le 1024 \times 1024\text{px}$ native optical resolution with $0\text{px}$ artificial gutters.
  - **Stage 6 Foveated VLM Reading**: Reads candidate crops/sub-canvases in a single batch without early consensus exit (`enable_early_consensus_exit=False`), extracting structured answers and consolidating into semicolon-separated natural language.


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, BRANCH SYNCHRONIZATION & HARDWARE TELEMETRY
# ==============================================================================
import os
import sys
import gc
import json
import re
import time
import subprocess
import importlib
import urllib.request
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

TARGET_BRANCH = "draft/improve_all"
REPO_DIR = "/kaggle/working/MSSearch" if os.path.exists("/kaggle") else os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(REPO_DIR, "mssearch")) and os.path.exists("mssearch"):
    REPO_DIR = os.path.abspath(os.getcwd())

# Retrieve GitHub Access Token from Kaggle User Secrets if in Kaggle
github_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception:
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if os.path.exists("/kaggle") and not os.path.exists(REPO_DIR):
    clone_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git" if github_token else "https://github.com/tothanhdat2006/MSSearch.git"
    print(f"[SETUP] Cloning MSSearch repository (branch: {TARGET_BRANCH}) into {REPO_DIR}...")
    ret = subprocess.run(["git", "clone", "-b", TARGET_BRANCH, clone_url, REPO_DIR], capture_output=True, text=True)
    if ret.returncode != 0:
        subprocess.run(["git", "clone", clone_url, REPO_DIR], capture_output=True, text=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", TARGET_BRANCH], capture_output=True, text=True)
elif os.path.exists(REPO_DIR) and os.path.exists("/kaggle"):
    print(f"[SETUP] Updating repository at {REPO_DIR} to branch {TARGET_BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "checkout", TARGET_BRANCH], capture_output=True, text=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", TARGET_BRANCH], capture_output=True, text=True)

# Inject REPO_DIR into sys.path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith("mssearch"):
        del sys.modules[mod]

# Verify BPE tokenizer asset for SAM3
bpe_target = os.path.join(REPO_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_target):
    os.makedirs(os.path.dirname(bpe_target), exist_ok=True)
    bpe_url = "https://raw.githubusercontent.com/openai/CLIP/main/clip/bpe_simple_vocab_16e6.txt.gz"
    try:
        urllib.request.urlretrieve(bpe_url, bpe_target)
        print("[SETUP] Successfully downloaded SAM3 BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download: {e}")

# Hardware Telemetry
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"\n[Hardware Telemetry]")
print(f"  - PyTorch Version : {torch.__version__}")
print(f"  - CUDA Available  : {torch.cuda.is_available()} (Count: {num_gpus})")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    total_mem = props.total_memory / (1024**3)
    print(f"  - GPU {i}           : {props.name} ({total_mem:.1f} GB VRAM)")


In [ ]:
# ==============================================================================
# CELL 2: MULTI-SOURCE DATASET INGESTION & N-IMAGE CANDIDATE POOL SAMPLING
# ==============================================================================
import os
import json
import random
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

HRBENCH_SEARCH_DIRS = [
    "D:/Research/VisualSearch/hr-bench_8k",
    "d:/Research/VisualSearch/hr-bench_8k",
    "D:/Research/VisualSearch/hr-bench_4k",
    "d:/Research/VisualSearch/hr-bench_4k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_4k",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    os.path.join(REPO_DIR, "data", "hr-bench_8k"),
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs/experiments/freeform/ICML26-CVSearch/data/hr-bench_8k"
]

HRBENCH_BASE_DIR = None
for d in HRBENCH_SEARCH_DIRS:
    if os.path.exists(d):
        HRBENCH_BASE_DIR = d
        break

if HRBENCH_BASE_DIR is None:
    raise FileNotFoundError(f"[Dataset Error] Could not find HRBench directory in paths: {HRBENCH_SEARCH_DIRS}")

HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_8k.json")
if not os.path.exists(HRBENCH_ANNOTATION_PATH):
    HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_4k.json")

print(f"[Dataset Discovery] Using HRBench root: {HRBENCH_BASE_DIR}")
print(f"[Dataset Discovery] Loading annotations: {HRBENCH_ANNOTATION_PATH}")

with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
    hrbench_data = json.load(f)

print(f"[Dataset] Ingested {len(hrbench_data)} total samples.")

# ------------------------------------------------------------------------------
# MULTI-SOURCE CONFIGURATION KNOBS
# ------------------------------------------------------------------------------
SAMPLE_INDEX = 0             # Primary target sample index
NUM_CANDIDATE_IMAGES = 3     # Total N candidate images (1 target + N-1 distractors)
WINDOW_SIZE = 5              # Local index sampling window: [SAMPLE_INDEX - 5, SAMPLE_INDEX + 5]
RANDOM_SEED = 42             # Deterministic sampling & shuffling seed

target_sample = hrbench_data[SAMPLE_INDEX]
real_question = target_sample.get('question', '')
cycle_idx = target_sample.get('cycle_category', 0)
gt_answer = target_sample.get('answer', '')

def resolve_gt_letter(answer_obj, cycle_idx=0):
    if isinstance(answer_obj, list):
        if 0 <= cycle_idx < len(answer_obj):
            return str(answer_obj[cycle_idx]).strip().upper()
        return str(answer_obj[0]).strip().upper() if answer_obj else ""
    return str(answer_obj).strip().upper()

def parse_options_text(options_raw, cycle_idx=0):
    opt_block = ""
    if isinstance(options_raw, list):
        if 0 <= cycle_idx < len(options_raw):
            opt_block = str(options_raw[cycle_idx])
        elif options_raw:
            opt_block = str(options_raw[0])
    elif isinstance(options_raw, dict):
        return {k.upper(): str(v).strip() for k, v in options_raw.items()}
    elif isinstance(options_raw, str):
        opt_block = options_raw
    out = {}
    for line in opt_block.splitlines():
        line = line.strip()
        m = re.match(r"^[\(\[\"']*([A-Da-d])[\)\]\.\:\s]+(.*)$", line)
        if m:
            out[m.group(1).upper()] = m.group(2).strip()
    return out

gt_letter = resolve_gt_letter(gt_answer, cycle_idx)
raw_options = target_sample.get('options', {})
parsed_options_dict = parse_options_text(raw_options, cycle_idx)

# Extract ground-truth bounding box if available
gt_bbox = target_sample.get("bbox") or target_sample.get("box") or target_sample.get("target_bbox")

print("=" * 75)
print(f"[Target Sample #{SAMPLE_INDEX}] Question: '{real_question}'")
print(f"[Target Sample #{SAMPLE_INDEX}] Ground Truth: Option '{gt_letter}' | Parsed Options: {parsed_options_dict}")
if gt_bbox:
    print(f"[Target Sample #{SAMPLE_INDEX}] Ground Truth BBox: {gt_bbox}")
print("=" * 75)

# Load Target Image
target_img_rel = target_sample.get('input_image', '')
target_img_path = os.path.join(HRBENCH_BASE_DIR, target_img_rel)
if not os.path.exists(target_img_path):
    target_img_path = target_img_rel
target_canvas = Image.open(target_img_path).convert("RGB")

# Sample Distractor Images
total_samples = len(hrbench_data)
min_win = max(0, SAMPLE_INDEX - WINDOW_SIZE)
max_win = min(total_samples - 1, SAMPLE_INDEX + WINDOW_SIZE)

candidate_distractor_indices = [idx for idx in range(min_win, max_win + 1) if idx != SAMPLE_INDEX]
if len(candidate_distractor_indices) < (NUM_CANDIDATE_IMAGES - 1):
    candidate_distractor_indices = [idx for idx in range(total_samples) if idx != SAMPLE_INDEX]

rng = random.Random(RANDOM_SEED + SAMPLE_INDEX)
sampled_distractor_indices = rng.sample(candidate_distractor_indices, NUM_CANDIDATE_IMAGES - 1)

pool_items = [
    {"is_target": True, "sample_idx": SAMPLE_INDEX, "image": target_canvas, "path": target_img_rel, "gt_bbox": gt_bbox}
]

for d_idx in sampled_distractor_indices:
    d_sample = hrbench_data[d_idx]
    d_rel = d_sample.get('input_image', '')
    d_path = os.path.join(HRBENCH_BASE_DIR, d_rel)
    if not os.path.exists(d_path):
        d_path = d_rel
    d_img = Image.open(d_path).convert("RGB")
    pool_items.append({
        "is_target": False, "sample_idx": d_idx, "image": d_img, "path": d_rel, "gt_bbox": None
    })

# Deterministic shuffle
rng.shuffle(pool_items)
candidate_images = [item["image"] for item in pool_items]
target_pool_idx = next(i for i, item in enumerate(pool_items) if item["is_target"])

print(f"[Candidate Pool C] Total N = {len(candidate_images)} images constructed.")
print(f"[Candidate Pool C] Target image is positioned at Pool Index: {target_pool_idx}")
for idx, item in enumerate(pool_items):
    tag = "TARGET" if item["is_target"] else "DISTRACTOR"
    print(f"  - Image {idx}: [{tag}] Sample #{item['sample_idx']} | Size: {item['image'].size}")

# ------------------------------------------------------------------------------
# Side-by-Side Thumbnail Visualization of Candidate Pool C
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(5.2 * len(candidate_images), 5.2))
if len(candidate_images) == 1:
    axes = [axes]

for i, (ax, item) in enumerate(zip(axes, pool_items)):
    thumb = item["image"].copy()
    thumb_w, thumb_h = thumb.size
    ax.imshow(thumb)
    is_t = item["is_target"]
    color = "#10b981" if is_t else "#3b82f6"
    tag = "TARGET" if is_t else "DISTRACTOR"
    
    # Overlay GT Bounding Box on Target Image if present
    if is_t and item["gt_bbox"]:
        bx1, by1, bx2, by2 = item["gt_bbox"]
        rect = mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=3.0, edgecolor='#f59e0b', facecolor='none', linestyle='-')
        ax.add_patch(rect)
        ax.text(bx1 + 10, by1 + 35, "Ground Truth BBox", fontsize=8.5, fontweight='bold', color='black',
                bbox=dict(boxstyle='square,pad=0.2', facecolor='#f59e0b', alpha=0.9))
        
    ax.set_title(f"Image {i} [{tag}]\nSample #{item['sample_idx']} | {thumb_w}x{thumb_h}px", fontsize=11, fontweight="bold", color=color)
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

plt.suptitle(f"Candidate Pool C (N = {len(candidate_images)}) for Question:\n'{real_question}'", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

active_canvas = candidate_images[target_pool_idx]


In [ ]:
# ==============================================================================
# CELL 3: FOUNDATION MODEL INITIALIZATION & MULTI-SOURCE CONFIGURATION
# ==============================================================================
import os
import sys
import gc
import torch
from PIL import Image

from mssearch.config import MSSearchConfig
from mssearch.stages.stage1_parser import parse_question_to_active_objects, ParsedQuestion
from mssearch.stages.stage2_fast_path import run_direct_global_assessment, DirectAnswerResult, run_tool_based_fast_path, FastPathResult
from mssearch.stages.stage3_partitioning import partition_candidate_images, BoundedPatch
from mssearch.stages.stage4_grouping_gating import run_feature_grouping_and_gating, group_patches_by_cosine_similarity
from mssearch.stages.stage5_screening import run_resolution_screening, CandidateCrop
from mssearch.stages.stage6_foveated_reader import run_foveated_vlm_reading, is_negative_or_unanswerable, deduplicate_answers
from mssearch.MSSearch import MultiSourceSearchEngine, get_mssearch_response

# 1. Discover SAM3 Weights
SAM3_SEARCH_PATHS = [
    "/kaggle/input/datasets/thnhtt/vs-sam3/sam3/sam3/sam3.pt",
    "/kaggle/input/sam3-weights/sam3.pt",
    "D:/Research/VisualSearch/sam3/sam3.pt",
    os.path.join(REPO_DIR, "assets", "sam3.pt")
]

SAM3_CHECKPOINT_PATH = None
for p in SAM3_SEARCH_PATHS:
    if os.path.exists(p):
        SAM3_CHECKPOINT_PATH = p
        break

if SAM3_CHECKPOINT_PATH is None and os.path.exists("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f == "sam3.pt":
                SAM3_CHECKPOINT_PATH = os.path.join(root, f)
                break
        if SAM3_CHECKPOINT_PATH:
            break

# 2. Hardware Allocation
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
sam3_device = "cuda:1" if num_gpus > 1 else ("cuda:0" if num_gpus > 0 else "cpu")
siglip_device = "cuda:0" if num_gpus > 0 else "cpu"
qwen_device = "cuda:0" if num_gpus > 0 else "cpu"

print(f"[Hardware Topology] Detected {num_gpus} GPUs. Allocation:")
print(f"  - SAM3 Engine      : {sam3_device} (Checkpoint: {SAM3_CHECKPOINT_PATH or 'Mock / Absent'})")
print(f"  - SigLIP-SO400M    : {siglip_device}")
print(f"  - Qwen2.5-VL Engine: {qwen_device}")

# 3. Initialize Models with Graceful Dry-Run Fallbacks
try:
    from mssearch.models.modeling_sam3 import sam3_inference
    if SAM3_CHECKPOINT_PATH and os.path.exists(SAM3_CHECKPOINT_PATH):
        print(f"[Loading] SAM3 Segmenter on {sam3_device}...")
        sam3_engine = sam3_inference(model_path=SAM3_CHECKPOINT_PATH, device=sam3_device)
        print("  [Loaded] SAM3 Local Segmenter Engine")
    else:
        sam3_engine = None
        print("  [Notice] SAM3 checkpoint not found; running without SAM3 segmenter or in mock mode.")
except Exception as e:
    print(f"  [Notice] SAM3 initialization skipped: {e}")
    sam3_engine = None

try:
    from mssearch.models.modeling_siglip import SigLIPScorer
    print(f"[Loading] SigLIP-SO400M on {siglip_device}...")
    siglip_engine = SigLIPScorer(model_name="google/siglip-base-patch16-384", device=siglip_device, torch_dtype=torch.float16 if "cuda" in siglip_device else torch.float32)
    print("  [Loaded] SigLIP Saliency & Verification Engine")
except Exception as e:
    print(f"  [Notice] SigLIP initialization skipped: {e}")
    siglip_engine = None

try:
    from mssearch.models.modeling_qwenvl import ModelQwenVL
    print(f"[Loading] Qwen2.5-VL-7B-Instruct on {qwen_device}...")
    qwen_engine = ModelQwenVL(model_path="Qwen/Qwen2.5-VL-7B-Instruct", device=qwen_device, torch_dtype=torch.float16 if "cuda" in qwen_device else torch.float32, patch_scale=1.2)
    print("  [Loaded] Qwen2.5-VL Multi-Scale Reader Engine")
except Exception as e:
    print(f"  [Notice] Qwen2.5-VL initialization skipped: {e}")
    qwen_engine = None

try:
    import spacy
    nlp_parser = spacy.load("en_core_web_sm")
    print("  [Loaded] spaCy NLP Grammar Engine")
except Exception:
    nlp_parser = None



# ------------------------------------------------------------------------------
# 4. Master Multi-Source Configuration
# ------------------------------------------------------------------------------
config_track_a = MSSearchConfig(
    # Multi-Source & Multi-Answer Knobs (Annotation Charter Standard)
    multi_answer_mode=True,               # Deactivate Stage 6 early exit; consolidate unique answers into semicolon text
    scale_budgets_by_num_images=False,    # Direct user budget enforcement

    # Phase 1 & 2 Early Assessment
    enable_direct_answer=True,
    direct_answer_conf_threshold=0.60,
    enable_fast_path=True,
    fast_path_conf_threshold=0.70,
    enable_visual_prompt_bbox=True,

    # Stage 3: Saliency-Plateau Mode with Quadtree Lineage Tracking
    partition_stopping_mode="saliency_plateau",
    min_plateau_size=256,
    max_plateau_size=1024,
    max_partition_depth=4,                # ponytail: 4 allows 8K images (8192px) to reach min_plateau_size 256px; use 3 if stopping at 1024px
    saliency_plateau_gain_threshold=0.04,
    saliency_target_threshold=0.35,
    max_patches_per_image=100,

    # Stage 4: Feature Grouping, Area-Ranked Gating & Recursive Re-Partitioning
    enable_feature_grouping=True,
    grouping_sim_threshold=0.80,
    enable_detection_bbox_area_ranking=True,
    enable_dynamic_retention_elasticity=True,
    enable_group_retention_caps=True,
    max_sam3_eval_patches=100,            # Stage 4 SAM3 scan budget
    max_latent_evals=20,                  # Speculative VLM latent queries
    max_patches_per_detected_group=4,     # Retained patches per confirmed detection group
    max_patches_per_non_detected_group=1, # Retained patches per latent group
    max_retained_patches=32,              # ponytail: 32 prevents candidate starvation across multi-source images; use 10 for strict single-image budget
    enable_latent_repartition=True,
    max_latent_repartition_patches_per_image=4,
    max_latent_repartition_depth=2,
    latent_repartition_threshold=0.70,    # ponytail: 0.70 prevents parent context pruning on weak latent noise; use 0.0 for aggressive feedback
    latent_prompt_mode="question_aware",

    # Stage 5: Continuous Enclosure Sub-Canvas Aggregation
    enable_continuous_enclosure_subcanvas=True,
    max_sub_canvases=4,
    seam_overlap_threshold=0.10,
    enable_spatial_compressed_canvas=True,
    spatial_compression_gutter_px=28,
    enable_stage5_spatial_nms=True,
    stage5_nms_iou_threshold=0.60,

    # Stage 6: Reading & De-duplication
    enable_early_consensus_exit=False,    # Evaluates all candidate crops across all N images
    prompt_directed_final_answer=True,
    max_vlm_tokens=128,
    answer_dedup_similarity=0.85,
    verbose=True
)

print("\n[CONFIG] MSSearch initialized with applied settings (N=" + str(NUM_CANDIDATE_IMAGES) + "):")
print(f"  - min_plateau_size / max_plateau_size  = {config_track_a.min_plateau_size} / {config_track_a.max_plateau_size}")
print(f"  - max_partition_depth                  = {config_track_a.max_partition_depth}")
print(f"  - max_patches_per_image                = {config_track_a.max_patches_per_image}")
print(f"  - max_sam3_eval_patches                = {config_track_a.max_sam3_eval_patches}")
print(f"  - max_latent_evals                     = {config_track_a.max_latent_evals}")
print(f"  - max_patches_per_detected_group       = {config_track_a.max_patches_per_detected_group}")
print(f"  - max_patches_per_latent_group         = {config_track_a.max_patches_per_non_detected_group}")
print(f"  - max_retained_patches                 = {config_track_a.max_retained_patches}")
print(f"  - enable_early_consensus_exit          = {config_track_a.enable_early_consensus_exit}")
print(f"  - max_latent_repartition_patches/depth = {config_track_a.max_latent_repartition_patches_per_image} / {config_track_a.max_latent_repartition_depth}")
print(f"  - latent_repartition_threshold         = {config_track_a.latent_repartition_threshold}")


In [ ]:
# ==============================================================================
# CELL 4: STAGE 1 - ZERO-SHOT MLLM QUESTION PARSING
# ==============================================================================
import time
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 1: QUESTION PARSING (Multi-Source Track A Sample {SAMPLE_INDEX})")
print(f"Input Question: '{real_question}'")
print(f"Candidate Images Pool C: {len(candidate_images)} images (Target at Pool Index {target_pool_idx})")
print(f"[Stage 1 Diagnostics] qwen_engine: {type(qwen_engine).__name__ if qwen_engine is not None else 'None'}")
print("=" * 75)

t0 = time.perf_counter()
raw_active_objects, parser_strategy = parse_question_to_active_objects(
    question=real_question,
    zoom_model=qwen_engine,
    nlp_model=nlp_parser,
    verbose=True,
    return_strategy=True,
    image_pil=candidate_images[target_pool_idx],
    config=config_track_a
)

active_objects = list(raw_active_objects.active_objects)
question_type = getattr(raw_active_objects, "question_type", "unknown")
is_relational = getattr(raw_active_objects, "is_relational", False)
latency_stage1 = round(time.perf_counter() - t0, 4)

print(f"\n[Stage 1 Output] Structured Semantic Parse:")
print(f"  - Question Type           : '{question_type}' (Relational: {is_relational})")
print(f"  - Final Active Objects A_q: {active_objects}")
print(f"[Stage 1 Telemetry] Extraction Strategy: {parser_strategy} | Latency: {latency_stage1}s\n")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 11)
ax.set_ylim(0, 3.4)
ax.axis('off')

card = mpatches.FancyBboxPatch((0.1, 0.2), 10.8, 3.0, boxstyle="round,pad=0.2", 
                               edgecolor='#1a237e', facecolor='#f8f9fa', linewidth=2.0)
ax.add_patch(card)

ax.text(0.3, 2.55, f"[Multi-Source Sample {SAMPLE_INDEX}] {real_question}", fontsize=11, fontweight='bold', color='#1a237e')
ax.text(0.3, 1.85, f"Type: {question_type} | Relational: {is_relational} | Strategy: {parser_strategy} | Pool Size: N={len(candidate_images)}", 
        fontsize=9.5, color='#424242', style='italic')

chips_x = 0.3
for i, obj in enumerate(active_objects):
    badge = 'Target' if i == 0 else 'Object'
    border_col = '#c2185b' if i == 0 else '#00796b'
    bg_col = '#fce4ec' if i == 0 else '#e0f2f1'
    txt_col = '#880e4f' if i == 0 else '#004d40'
    
    chip = mpatches.FancyBboxPatch((chips_x, 0.45), len(obj)*0.20 + 1.05, 0.76, 
                                   boxstyle="round,pad=0.1", edgecolor=border_col, facecolor=bg_col, linewidth=1.5)
    ax.add_patch(chip)
    ax.text(chips_x + 0.15, 0.65, f"{badge}: {obj}", fontsize=9.5, fontweight='bold', color=txt_col)
    chips_x += len(obj)*0.20 + 1.35

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 5: 2-GATE EARLY ASSESSMENT ACROSS ALL N CANDIDATE IMAGES
# ==============================================================================
print("=" * 75)
print(f"2-GATE EARLY ASSESSMENT ACROSS ALL N = {len(candidate_images)} CANDIDATE IMAGES")
print(f"Active Objects A_q: {active_objects}")
print(f"Phase 1 Threshold: {config_track_a.direct_answer_conf_threshold:.2f} | Phase 2 Threshold: {config_track_a.fast_path_conf_threshold:.2f}")
print("=" * 75)

# --- GATE 1: Phase 1 Direct Global Answer Assessment ---
t0_da = time.perf_counter()
direct_ans_res = run_direct_global_assessment(
    candidate_images=candidate_images,
    question=real_question,
    zoom_model=qwen_engine,
    config=config_track_a
)
t_da = time.perf_counter() - t0_da
print(f"\n[Phase 1 Gate Result] (Latency: {t_da:.3f}s)")
print(f"  - Triggered      : {direct_ans_res.triggered}")
print(f"  - Confidence     : {direct_ans_res.confidence:.3f}")
print(f"  - Answer(s)      : {direct_ans_res.answer}")
print(f"  - Evidence Images: {getattr(direct_ans_res, 'evidence_images', [direct_ans_res.evidence_img_idx])}")

# --- GATE 2: Stage 2 Tool-Based Fast Path Assessment ---
t0_fp = time.perf_counter()
fast_path_res = run_tool_based_fast_path(
    candidate_images=candidate_images,
    active_objects=active_objects,
    question=real_question,
    sam_model=sam3_engine,
    zoom_model=qwen_engine,
    config=config_track_a
)
t_fp = time.perf_counter() - t0_fp
print(f"\n[Stage 2 Fast Path Result] (Latency: {t_fp:.3f}s)")
print(f"  - Triggered      : {fast_path_res.triggered}")
print(f"  - Confidence     : {fast_path_res.confidence:.3f}")
print(f"  - Answer(s)      : {fast_path_res.answer}")
print(f"  - Evidence Images: {getattr(fast_path_res, 'evidence_images', [fast_path_res.evidence_img_idx])}")
print(f"  - Cached Features: {list(fast_path_res.feature_cache.keys())}")


In [ ]:
# ==============================================================================
# CELL 6: STAGE 3 - MULTI-IMAGE SALIENCY-PLATEAU PARTITIONING & LINEAGE TRACKING
# ==============================================================================
import copy
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 3: MULTI-IMAGE SALIENCY-PLATEAU PARTITIONING (N = {len(candidate_images)} Images)")
print(f"Stopping Mode: '{config_track_a.partition_stopping_mode}' (gain threshold: {config_track_a.saliency_plateau_gain_threshold})")
print("=" * 75)

t0_stage3 = time.perf_counter()
feature_cache = fast_path_res.feature_cache if config_track_a.reuse_stage2_features else {}

bounded_patches = partition_candidate_images(
    candidate_images=candidate_images,
    stage2_feature_cache=feature_cache,
    config=config_track_a,
    active_objects=active_objects,
    siglip_model=siglip_engine
)
t_stage3 = time.perf_counter() - t0_stage3

print(f"\n[Stage 3 Summary]")
print(f"  - Total Bounded Patches Generated : {len(bounded_patches)}")
print(f"  - Stage 3 Partitioning Latency    : {t_stage3:.3f}s")

# Distribution & lineage audit per candidate image
patches_per_img = {}
for p in bounded_patches:
    patches_per_img[p.img_idx] = patches_per_img.get(p.img_idx, 0) + 1

for img_idx in range(len(candidate_images)):
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    print(f"  - Image {img_idx} [{tag}]: {patches_per_img.get(img_idx, 0)} surviving candidate patches")

# ------------------------------------------------------------------------------
# 1. Visualize Partitioned Patches Overlaid Across All N Candidate Images
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6.2 * len(candidate_images), 6.2))
if len(candidate_images) == 1:
    axes = [axes]

cols = plt.cm.tab20(np.linspace(0, 1, max(1, len(bounded_patches))))
for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    ax.imshow(vis_img)
    img_patches = [(idx, p) for idx, p in enumerate(bounded_patches) if p.img_idx == img_idx]
    
    for idx, p in img_patches:
        x1, y1, x2, y2 = [int(round(float(c))) for c in p.box]
        c = cols[idx % len(cols)]
        rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.5, edgecolor=c, facecolor='none')
        ax.add_patch(rect)
        s_lbl = f"S={p.siglip_score:.2f}\n" if p.siglip_score is not None else ""
        lvl_lbl = f"L{getattr(p, 'level', 0)}"
        ax.text(x1 + 10, y1 + 35, f"P{idx} [{lvl_lbl}]\n{s_lbl}H={p.entropy:.2f}", fontsize=7.5, fontweight='bold',
                color='white', bbox=dict(boxstyle='square,pad=0.2', facecolor=c, alpha=0.85))
    
    is_t = (img_idx == target_pool_idx)
    color = "#10b981" if is_t else "#3b82f6"
    tag = "TARGET" if is_t else "DISTRACTOR"
    
    # Overlay Ground Truth BBox on Target Image
    if is_t and gt_bbox:
        bx1, by1, bx2, by2 = gt_bbox
        gt_rect = mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=3.5, edgecolor='#f59e0b', facecolor='none', linestyle='--')
        ax.add_patch(gt_rect)
        ax.text(bx1 + 10, by1 - 15, "GT Target Evidence", fontsize=8.5, fontweight='bold', color='#f59e0b',
                bbox=dict(boxstyle='square,pad=0.2', facecolor='black', alpha=0.8))
        
    ax.set_title(f"Image {img_idx} [{tag}]\n{len(img_patches)} Patches Generated",
                 fontsize=11, fontweight="bold", color=color)
    ax.axis("off")

plt.suptitle(f"Stage 3: Partitioned Candidate Patches Across Candidate Pool C (Total: {len(bounded_patches)})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. Candidate Patches Gallery (Crops Across Candidate Images)
# ------------------------------------------------------------------------------
n_show_s3 = min(len(bounded_patches), 12)
if n_show_s3 > 0:
    ncols = min(4, n_show_s3)
    nrows = (n_show_s3 + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.8 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    for i in range(n_show_s3):
        ax = axes[i]
        p = bounded_patches[i]
        is_t = (p.img_idx == target_pool_idx)
        crop = candidate_images[p.img_idx].crop(tuple(int(round(float(c))) for c in p.box))
        ax.imshow(crop)
        border_col = '#10b981' if is_t else '#3b82f6'
        tag = "TARGET" if is_t else "DISTRACTOR"
        s_lbl = f" | S: {p.siglip_score:.2f}" if p.siglip_score is not None else ""
        lvl = getattr(p, 'level', 0)
        ax.set_title(f"P{i} [Img {p.img_idx} - {tag}] (L{lvl})\nSize: {crop.width}x{crop.height}px | H: {p.entropy:.2f}{s_lbl}",
                     fontsize=8.5, fontweight='bold', color=border_col)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col)
            spine.set_linewidth(2.5)
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_show_s3, len(axes)):
        axes[j].axis('off')
        
    plt.suptitle(f"Stage 3: Candidate Patches Gallery (Showing first {n_show_s3} of {len(bounded_patches)})", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# CELL 7: STAGE 4 - SAM3 FEATURE GROUPING & AREA-RANKED ACTIVE OBJECT GATING
# ==============================================================================
import textwrap
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 4: SAM3 FEATURE GROUPING & SCALED GATING (N = {len(candidate_images)} Images)")
print(f"Budgets: max_sam3_eval_patches = {config_track_a.max_sam3_eval_patches} | max_latent_evals = {config_track_a.max_latent_evals} | max_retained = {config_track_a.max_retained_patches}")
print(f"Ranking Strategy: Detection BBox Area Descending (max w*h), Entropy Purged, Dynamic Retention Elasticity Enabled")
print("=" * 75)

t0_stage4 = time.perf_counter()
remaining_patches, stage4_groups = run_feature_grouping_and_gating(
    bounded_patches=bounded_patches,
    candidate_images=candidate_images,
    active_objects=active_objects,
    sam_model=sam3_engine,
    config=config_track_a,
    detected_boxes=fast_path_res.detected_boxes,
    zoom_model=qwen_engine,
    return_debug=True,
    question=real_question
)
t_stage4 = time.perf_counter() - t0_stage4

pruned_count = len(bounded_patches) - len(remaining_patches)
prune_pct = (pruned_count / max(1, len(bounded_patches))) * 100

print(f"\n[Stage 4 Summary]")
print(f"  - Total Formed Groups             : {len(stage4_groups)}")
print(f"  - Surviving Patches for Reading   : {len(remaining_patches)} / {len(bounded_patches)} ({pruned_count} pruned, {prune_pct:.1f}% reduction)")
print(f"  - Stage 4 Execution Latency       : {t_stage4:.3f}s")

# Breakdown of surviving patches per candidate image
surviving_per_img = {}
for p in remaining_patches:
    surviving_per_img[p.img_idx] = surviving_per_img.get(p.img_idx, 0) + 1

for img_idx in range(len(candidate_images)):
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    total_s3_img = len([p for p in bounded_patches if p.img_idx == img_idx])
    print(f"  - Image {img_idx} [{tag}]: {surviving_per_img.get(img_idx, 0)} / {total_s3_img} patches retained after SAM3/VLM gating")

# ------------------------------------------------------------------------------
# 1. Visualize Original Patches Kept vs. Pruned Across All N Candidate Images
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6.2 * len(candidate_images), 6.2))
if len(candidate_images) == 1:
    axes = [axes]

retained_s4_boxes = {(p.img_idx, tuple([int(round(float(c))) for c in p.box])) for p in remaining_patches}

for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    ax.imshow(vis_img)
    img_patches = [(idx, p) for idx, p in enumerate(bounded_patches) if p.img_idx == img_idx]
    
    n_k = 0
    n_p = 0
    for idx, p in img_patches:
        box_int = [int(round(float(c))) for c in p.box]
        x1, y1, x2, y2 = box_int
        is_kept = (img_idx, tuple(box_int)) in retained_s4_boxes
        
        if is_kept:
            n_k += 1
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=3.5, edgecolor='#00e676', facecolor='none')
            ax.add_patch(rect)
            s_lbl = f"S={p.siglip_score:.2f}\n" if p.siglip_score is not None else ""
            conf_lbl = f"C={p.confidence_score:.2f}\n" if getattr(p, 'confidence_score', None) is not None else ""
            ax.text(x1 + 10, y1 + 35, f"P{idx}\n{s_lbl}{conf_lbl}KEPT", fontsize=7.5, fontweight='bold',
                    color='black', bbox=dict(boxstyle='square,pad=0.2', facecolor='#00e676', alpha=0.9))
        else:
            n_p += 1
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.8, edgecolor='#ef5350', facecolor='none', linestyle='--', alpha=0.7)
            ax.add_patch(rect)
            ax.text(x1 + 10, y1 + 25, f"P{idx}\nPRUNED", fontsize=6.5, fontweight='bold',
                    color='white', bbox=dict(boxstyle='square,pad=0.15', facecolor='#c62828', alpha=0.6))
    
    is_t = (img_idx == target_pool_idx)
    color = "#10b981" if is_t else "#3b82f6"
    label = f"Image {img_idx} [{'TARGET' if is_t else 'DISTRACTOR'}]\nKept: {n_k} | Pruned: {n_p} (from {len(img_patches)} total)"
    
    # Overlay GT BBox on Target Image
    if is_t and gt_bbox:
        bx1, by1, bx2, by2 = gt_bbox
        gt_rect = mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=3.5, edgecolor='#f59e0b', facecolor='none', linestyle='-')
        ax.add_patch(gt_rect)
        ax.text(bx1 + 10, by1 - 15, "GT Target Evidence", fontsize=8.5, fontweight='bold', color='#f59e0b',
                bbox=dict(boxstyle='square,pad=0.2', facecolor='black', alpha=0.8))
        
    ax.set_title(label, fontsize=11, fontweight="bold", color=color)
    ax.axis("off")

plt.suptitle(f"Stage 4: Original Patches Kept vs. Pruned Across Candidate Pool C\n(Solid Green: Retained {len(remaining_patches)} | Red Dashed: Pruned {pruned_count})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. Detailed Semantic Group Diagnostics (Micro-Fovea, Gating & Heatmaps)
# ------------------------------------------------------------------------------
if len(stage4_groups) == 0:
    print("[Stage 4 Notice] No semantic groups formed (0 candidate patches reached Stage 4).")
else:
    print(f"\n[Stage 4 Group Diagnostics] Rendering {len(stage4_groups)} semantic groups...")
    for g in stage4_groups[:6]:
        k = g["num_patches"]
        member_indices = g.get("member_indices", list(range(k)))
        member_boxes = g.get("member_boxes", [g.get("rep_box", [0, 0, 100, 100])])
        member_entropies = g.get("member_entropies", [0.5] * k)
        sim_mat = np.array(g.get("pairwise_sim", np.ones((k, k))))
        status = g["status"]
        reason_label = g.get("match_reason", "").replace('_', ' ').title()
        det_objects = g.get("detected_objects", [])
        vlm_exist_score = g.get("vlm_exist_score", None)
        vlm_latent_score = g.get("vlm_latent_score", None)
        max_det_score = g.get("max_det_score", 0.0)
        vlm_tested_patch_idx = g.get("vlm_tested_patch_idx", None)
        vlm_prompt = g.get("vlm_prompt", "")
        
        rep_idx = g.get("rep_idx", 0)
        source_img_idx = g.get("img_idx", bounded_patches[rep_idx].img_idx if rep_idx < len(bounded_patches) else 0)
        is_t_group = (source_img_idx == target_pool_idx)
        
        fovea_box = g.get("fovea_box", None)
        best_det_box = g.get("best_det_box", None)
        has_fovea = (fovea_box is not None)
        status_color = "#1b5e20" if status == "KEPT" else "#b71c1c"
        
        n_cols = 1 + k + (1 if has_fovea else 0) + 1
        col_ratios = [2.6] + [1.0] * k + ([1.1] if has_fovea else []) + [1.8]
        total_w = 5.4 + k * 2.3 + (2.6 if has_fovea else 0.0) + 3.0
        
        fig = plt.figure(figsize=(total_w, 5.2))
        fig.patch.set_facecolor('#ffffff')
        gs = fig.add_gridspec(1, n_cols, width_ratios=col_ratios, wspace=0.28)
        
        # 1. Left Card: Group Info, Telemetry
        ax_info = fig.add_subplot(gs[0])
        ax_info.axis('off')
        card = mpatches.FancyBboxPatch((0.02, 0.02), 0.96, 0.96, boxstyle="round,pad=0.05",
                                       edgecolor=status_color, facecolor='#fbfbfb', linewidth=2.5)
        ax_info.add_patch(card)
        
        ax_info.text(0.06, 0.92, f"GROUP {g['group_idx']}  [{status}]", fontsize=10.5, fontweight='bold', color=status_color)
        ax_info.text(0.06, 0.85, f"Patches: {k} | Source Img: {source_img_idx} [{'TARGET' if is_t_group else 'DISTRACTOR'}]",
                     fontsize=8.0, fontweight='bold', color='#1565c0')
        ax_info.text(0.06, 0.78, f"Evaluated Patch: P{vlm_tested_patch_idx if vlm_tested_patch_idx is not None else 'N/A'}",
                     fontsize=8.0, fontweight='bold', color='#333333')
        
        ax_info.text(0.06, 0.70, "[Micro-Fovea Token Boost]", fontsize=8.0, fontweight='bold', color='#6a1b9a')
        if fovea_box:
            fw = int(round(fovea_box[2] - fovea_box[0]))
            fh = int(round(fovea_box[3] - fovea_box[1]))
            ax_info.text(0.06, 0.63, f"- Foveated Window: {fw}x{fh}px", fontsize=7.4, color='#4a148c')
            ax_info.text(0.06, 0.56, "- Target Density: ~40% (vs 0.08%)", fontsize=7.4, color='#2e7d32', fontweight='bold')
        else:
            ax_info.text(0.06, 0.63, "- Full Patch View (Centroid)", fontsize=7.4, color='#616161')
            ax_info.text(0.06, 0.56, "- Token Density: Uniform", fontsize=7.4, color='#616161')
        
        ax_info.text(0.06, 0.48, "[VLM Gating Outcome]", fontsize=8.0, fontweight='bold', color='#0277bd')
        active_score = vlm_exist_score if vlm_exist_score is not None else vlm_latent_score
        score_label = "c_exist" if vlm_exist_score is not None else ("c_latent" if vlm_latent_score is not None else "SAM3 Det")
        score_str = f"{active_score:+.2f}" if active_score is not None else f"{max_det_score:.2f}"
        ax_info.text(0.06, 0.41, f"- {score_label}: {score_str}", fontsize=8.0, fontweight='bold', color='#1565c0')
        
        if vlm_prompt:
            ax_info.text(0.06, 0.33, "- Prompt Asked:", fontsize=7.4, fontweight='bold', color='#37474f')
            wrapped_q = textwrap.fill(vlm_prompt[:95] + ('...' if len(vlm_prompt) > 95 else ''), width=26)
            ax_info.text(0.06, 0.17, f'"{wrapped_q}"', fontsize=6.8, style='italic', color='#546e7a')
        else:
            ax_info.text(0.06, 0.33, f"- Match: {reason_label[:28]}", fontsize=7.4, color='#546e7a')
            
        # 2. Middle Columns: Group Member Patch Crops
        for col_idx, mem_p_idx in enumerate(member_indices):
            ax_p = fig.add_subplot(gs[1 + col_idx])
            p_box = [int(round(float(c))) for c in member_boxes[col_idx]]
            crop_p = candidate_images[source_img_idx].crop(p_box)
            ax_p.imshow(crop_p)
            
            # Draw local detections
            for d in det_objects:
                if d.get("patch_idx") == mem_p_idx:
                    lb = d.get("local_box")
                    if lb:
                        bx1, by1, bx2, by2 = [int(round(float(c))) for c in lb]
                        rect = mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=2.0,
                                                  edgecolor='#00e676', facecolor='none')
                        ax_p.add_patch(rect)
                        
            is_rep = (mem_p_idx == g.get("rep_idx"))
            border_col = '#ff9800' if is_rep else ('#2e7d32' if status == 'KEPT' else '#b71c1c')
            title_txt = f"P{mem_p_idx} {'[CENTROID]' if is_rep else ''}\nH={member_entropies[col_idx]:.2f}"
            ax_p.set_title(title_txt, fontsize=8.0, fontweight='bold', color=border_col)
            ax_p.set_xticks([])
            ax_p.set_yticks([])
            for spine in ax_p.spines.values():
                spine.set_edgecolor(border_col)
                spine.set_linewidth(2.5 if is_rep else 1.5)
                
        # 3. Micro-Fovea Zoom Window
        if has_fovea and fovea_box:
            ax_fov = fig.add_subplot(gs[1 + k])
            rep_patch_crop = candidate_images[source_img_idx].crop(tuple(member_boxes[0]))
            fb_ints = [int(round(float(c))) for c in fovea_box]
            fovea_img = rep_patch_crop.crop(tuple(fb_ints))
            ax_fov.imshow(fovea_img)
            if best_det_box:
                rel_x1 = max(0, int(round(best_det_box[0] - fb_ints[0])))
                rel_y1 = max(0, int(round(best_det_box[1] - fb_ints[1])))
                rel_x2 = min(fovea_img.width, int(round(best_det_box[2] - fb_ints[0])))
                rel_y2 = min(fovea_img.height, int(round(best_det_box[3] - fb_ints[1])))
                r = mpatches.Rectangle((rel_x1, rel_y1), rel_x2 - rel_x1, rel_y2 - rel_y1,
                                       linewidth=2.0, edgecolor='#00e676', facecolor='none')
                ax_fov.add_patch(r)
            ax_fov.set_title("[Micro-Fovea]\nHigh Density Crop", fontsize=8.0, fontweight='bold', color='#6a1b9a')
            ax_fov.set_xticks([])
            ax_fov.set_yticks([])
            for spine in ax_fov.spines.values():
                spine.set_edgecolor('#ab47bc')
                spine.set_linewidth(2.0)
                
        # 4. Right: Pairwise Cosine Similarity Heatmap
        ax_hm = fig.add_subplot(gs[-1])
        im = ax_hm.imshow(sim_mat, cmap='YlGnBu', vmin=0.5, vmax=1.0)
        ax_hm.set_title(f"Cosine Sim (k={k})", fontsize=8.0, fontweight='bold')
        ax_hm.set_xticks(range(k))
        ax_hm.set_yticks(range(k))
        ax_hm.set_xticklabels([f"P{p}" for p in member_indices], fontsize=6.5)
        ax_hm.set_yticklabels([f"P{p}" for p in member_indices], fontsize=6.5)
        for r in range(k):
            for c in range(k):
                val = sim_mat[r, c]
                text_col = 'white' if val > 0.85 else 'black'
                ax_hm.text(c, r, f"{val:.2f}", ha='center', va='center', fontsize=6.5, color=text_col, fontweight='bold')
        cbar = fig.colorbar(im, ax=ax_hm, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=6)
        
        plt.subplots_adjust(top=0.88, bottom=0.08, left=0.02, right=0.98)
        plt.show()


In [ ]:
# ==============================================================================
# CELL 7B: STAGE 4 - ALL SAM3 OBJECT DETECTIONS OVERLAID ACROSS CANDIDATE POOL C
# ==============================================================================
import textwrap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("=" * 75)
print("STAGE 4: ALL SAM3 DETECTED OBJECT BOUNDING BOXES (GLOBAL OVERLAY)")
print(f"Target Pool Index: {target_pool_idx} | Total Formed Groups: {len(stage4_groups)}")
print("=" * 75)

# 1. Aggregate all detected objects across groups with global coordinates
all_detections_by_img = {i: [] for i in range(len(candidate_images))}

for g in stage4_groups:
    g_idx = g.get("group_idx", 0)
    g_status = g.get("status", "UNKNOWN")
    for d in g.get("detected_objects", []):
        p_idx = d.get("patch_idx")
        if p_idx is not None and p_idx < len(bounded_patches):
            img_idx = bounded_patches[p_idx].img_idx
            all_detections_by_img[img_idx].append({
                "group_idx": g_idx,
                "group_status": g_status,
                "patch_idx": p_idx,
                "label": d.get("label", "Object"),
                "score": d.get("score", 0.0),
                "global_box": d.get("global_box", [0, 0, 0, 0]),
                "local_box": d.get("local_box", [0, 0, 0, 0]),
                "source": d.get("source", "sam3_crop"),
            })

total_dets = sum(len(dets) for dets in all_detections_by_img.values())
print(f"\n[Stage 4 Detection Audit] Extracted {total_dets} total SAM3 object detections across {len(candidate_images)} images:")
for img_idx in range(len(candidate_images)):
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    print(f"  - Image {img_idx} [{tag}]: {len(all_detections_by_img[img_idx])} detected objects")

# 2. Side-by-Side Visualization Overlaid on Full Candidate Images
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6.5 * len(candidate_images), 6.5))
if len(candidate_images) == 1:
    axes = [axes]

for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    ax.imshow(vis_img)
    
    img_dets = all_detections_by_img[img_idx]
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    title_color = "#10b981" if is_t else "#3b82f6"
    
    for i, d in enumerate(img_dets):
        gx1, gy1, gx2, gy2 = [int(round(float(c))) for c in d["global_box"]]
        bw = max(1, gx2 - gx1)
        bh = max(1, gy2 - gy1)
        is_kept_g = (d["group_status"] == "KEPT")
        
        box_color = '#00e676' if is_kept_g else '#ff9100'
        rect = mpatches.Rectangle((gx1, gy1), bw, bh, linewidth=2.8, edgecolor=box_color, facecolor='none')
        ax.add_patch(rect)
        
        lbl_txt = f"{d['label']}\n{d['score']:.2f} [G{d['group_idx']}]"
        ax.text(gx1 + 6, gy1 + 22, lbl_txt, fontsize=7.2, fontweight='bold', color='black',
                bbox=dict(boxstyle='square,pad=0.15', facecolor=box_color, alpha=0.9))
        
    # Overlay Ground Truth BBox on Target Image
    if is_t and gt_bbox:
        bx1, by1, bx2, by2 = gt_bbox
        gt_rect = mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=3.5, edgecolor='#f59e0b', facecolor='none', linestyle='-')
        ax.add_patch(gt_rect)
        ax.text(bx1 + 10, by1 - 15, "GT Target Evidence", fontsize=8.5, fontweight='bold', color='#f59e0b',
                bbox=dict(boxstyle='square,pad=0.2', facecolor='black', alpha=0.8))
        
    ax.set_title(f"Image {img_idx} [{tag}]\n{len(img_dets)} Detected Objects Overlaid", fontsize=11, fontweight="bold", color=title_color)
    ax.axis("off")

plt.suptitle(f"Stage 4: All Detected Objects Overlaid on Candidate Pool C (Total: {total_dets})\n(Green: Retained in Kept Groups | Orange: Pruned with Dropped Groups)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 8: STAGE 5 - CONTINUOUS ENCLOSURE SUB-CANVAS AGGREGATION ACROSS N IMAGES
# ==============================================================================
print("=" * 75)
print("STAGE 5: CONTINUOUS ENCLOSURE SUB-CANVAS AGGREGATION ACROSS N IMAGES")
print(f"Knobs: enable_continuous_enclosure={config_track_a.enable_continuous_enclosure_subcanvas} | max_sub_canvases={config_track_a.max_sub_canvases} | max_dim=1024px | relational={is_relational}")
print("=" * 75)

t0_stage5 = time.perf_counter()
unified_pool = run_resolution_screening(
    remaining_patches=remaining_patches,
    candidate_images=candidate_images,
    active_objects=active_objects,
    siglip_model=siglip_engine,
    config=config_track_a,
    is_cross_object=False,
    is_relational=is_relational
)
t_stage5 = time.perf_counter() - t0_stage5

print(f"\n[Stage 5 Summary]")
print(f"  - Unified Candidate Pool P Size   : {len(unified_pool)} items (Sub-canvases / Native Crops)")
print(f"  - Stage 5 Screening Latency       : {t_stage5:.3f}s")

# Distribution in unified pool
pool_per_img = {}
for c in unified_pool:
    pool_per_img[c.img_idx] = pool_per_img.get(c.img_idx, 0) + 1

for img_idx in range(len(candidate_images)):
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    print(f"  - Image {img_idx} [{tag}]: {pool_per_img.get(img_idx, 0)} candidate crops ready for Stage 6 reading")

# ------------------------------------------------------------------------------
# 1. Visualize Retained Sub-Canvases / Crops in Unified Candidate Pool P
# ------------------------------------------------------------------------------
n_show = min(len(unified_pool), 8)
if n_show > 0:
    ncols = min(4, n_show)
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.4 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    fig.suptitle(f"Stage 5: Unified Candidate Pool P ({len(unified_pool)} Total Retained Sub-Canvases / Crops)\n(Continuous Enclosure Bounded strictly to <= 1024x1024px with 0px Artificial Gutters)", 
                 fontsize=12, fontweight='bold', y=1.02)
    
    for i in range(n_show):
        ax = axes[i]
        cand = unified_pool[i]
        is_t = (cand.img_idx == target_pool_idx)
        crop = cand.crop_image if hasattr(cand, 'crop_image') and cand.crop_image is not None else candidate_images[cand.img_idx].crop(cand.box)
        ax.imshow(crop)
        
        # Overlay remapped candidate boxes inside sub-canvas
        sub_boxes = getattr(cand, 'subcanvas_boxes', [])
        for sb in sub_boxes:
            bx1, by1, bx2, by2 = sb
            rect = mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=2.0, edgecolor='#00e5ff', facecolor='none')
            ax.add_patch(rect)
            
        score_val = getattr(cand, 'confidence_score', getattr(cand, 'score', 1.0))
        branch = getattr(cand, 'source_branch', 'ContinuousEnclosure')
        tag = "TARGET" if is_t else "DISTRACTOR"
        border_color = '#10b981' if is_t else '#3b82f6'
        
        ax.set_title(f"Item {i} | Image {cand.img_idx} [{tag}]\nSize: {crop.width}x{crop.height}px | Branch: {branch}\nSub-boxes: {len(sub_boxes)}", 
                     fontsize=8.5, fontweight='bold', color=border_color)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2.5)
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------
# 2. Global Overlay of Continuous Enclosure Windows on Source Candidate Images
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6.2 * len(candidate_images), 6.2))
if len(candidate_images) == 1:
    axes = [axes]

for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    ax.imshow(vis_img)
    
    img_cands = [c for c in unified_pool if c.img_idx == img_idx]
    for c in img_cands:
        gx1, gy1, gx2, gy2 = [int(round(float(v))) for v in c.box]
        rect = mpatches.Rectangle((gx1, gy1), gx2 - gx1, gy2 - gy1, linewidth=3.5, edgecolor='#00e5ff', facecolor='none')
        ax.add_patch(rect)
        ax.text(gx1 + 10, gy1 + 35, f"SubCanvas\n{gx2-gx1}x{gy2-gy1}px", fontsize=7.5, fontweight='bold', color='black',
                bbox=dict(boxstyle='square,pad=0.2', facecolor='#00e5ff', alpha=0.9))
        
    is_t = (img_idx == target_pool_idx)
    title_color = "#10b981" if is_t else "#3b82f6"
    tag = "TARGET" if is_t else "DISTRACTOR"
    
    if is_t and gt_bbox:
        bx1, by1, bx2, by2 = gt_bbox
        gt_rect = mpatches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=3.5, edgecolor='#f59e0b', facecolor='none', linestyle='-')
        ax.add_patch(gt_rect)
        ax.text(bx1 + 10, by1 - 15, "GT Target Evidence", fontsize=8.5, fontweight='bold', color='#f59e0b',
                bbox=dict(boxstyle='square,pad=0.2', facecolor='black', alpha=0.8))
        
    ax.set_title(f"Image {img_idx} [{tag}]\n{len(img_cands)} Continuous Enclosure Windows", fontsize=11, fontweight="bold", color=title_color)
    ax.axis("off")

plt.suptitle(f"Stage 5: Continuous Enclosure Sub-Canvas Windows Overlaid Across Candidate Pool C", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 9: STAGE 6 - FOVEATED VLM READING & OPEN-SET MULTI-ANSWER DEDUPLICATION
# ==============================================================================
import textwrap
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 6: FOVEATED VLM READING WITHOUT EARLY CONSENSUS EXIT (Sample {SAMPLE_INDEX})")
print(f"Question: '{real_question}' | Multi-Answer Mode: {config_track_a.multi_answer_mode}")
print("=" * 75)

if qwen_engine is None:
    print("\n[CRITICAL NOTICE] qwen_engine (Qwen2.5-VL) is None (Dry-Run / CPU mode).")
    print("  -> Stage 6 VLM forward pass requires GPU. Testing reader pipeline logic with fallback...")

t0_stage6 = time.perf_counter()
final_result = run_foveated_vlm_reading(
    unified_pool=unified_pool,
    candidate_images=candidate_images,
    question=real_question,
    zoom_model=qwen_engine,
    config=config_track_a,
    question_type=question_type
)
t_stage6 = time.perf_counter() - t0_stage6

# Prioritize informative answers over refusal / negative answers
from mssearch.stages.stage6_foveated_reader import is_negative_or_unanswerable
informative_answers = [
    a for a in final_result.answers if not is_negative_or_unanswerable(a)
]

if config_track_a.multi_answer_mode:
    consolidated_answer = "; ".join(informative_answers) if informative_answers else "; ".join(final_result.answers) if final_result.answers else "No answer found"
else:
    consolidated_answer = informative_answers[0] if informative_answers else final_result.answers[0] if final_result.answers else "No answer found"

crop_records = getattr(final_result, "crop_records", [])

# Multiple Choice Option Letter Resolution
def extract_multiple_choice_letter(pred_text, options_dict):
    if not pred_text:
        return None
    clean = pred_text.strip()
    # Exact letter check
    m = re.match(r"^[\(\[\"']*([A-Da-d])[\)\]\.\:\s]", clean)
    if m:
        return m.group(1).upper()
    if clean.upper() in ["A", "B", "C", "D"]:
        return clean.upper()
    # Keyword match
    m2 = re.search(r"\b(?:final answer|answer|choice|option)\b[^\w\n\r]*([A-Da-d])\b", clean, re.IGNORECASE)
    if m2:
        return m2.group(1).upper()
    # Text content matching
    p_low = clean.lower()
    for letter, text in options_dict.items():
        if text and (p_low == text.lower() or text.lower() in p_low):
            return letter
    return None

predicted_letter = extract_multiple_choice_letter(consolidated_answer, parsed_options_dict)
is_correct = (predicted_letter == gt_letter) if predicted_letter and gt_letter else False

print(f"\n[Stage 6 Multi-Source Reading Results]")
print(f"  - Total Crops Evaluated           : {len(crop_records)} / {len(unified_pool)}")
print(f"  - Raw Extractions Count           : {len(final_result.extractions)}")
print(f"  - Structured Unique Answers (List): {final_result.answers}")
print(f"  - Informative Filtered Answers    : {informative_answers}")
print(f"  - Consolidated Multi-Answer Text  : '{consolidated_answer}'")
print(f"  - Predicted Option Letter         : '{predicted_letter}' (Ground Truth: '{gt_letter}') -> {'[CORRECT]' if is_correct else '[INCORRECT]'}")
print(f"  - Discovered Evidence Images      : {final_result.evidence_image_indices} (Target was at Pool Index {target_pool_idx})")
print(f"  - Stage 6 Reading Latency         : {t_stage6:.3f}s")

# Display each extraction detail
print(f"\n[Evidence Extractions Detail]")
for i, ext in enumerate(final_result.extractions):
    is_t = (ext.img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    print(f"  - Extraction #{i+1} [Image {ext.img_idx} - {tag}]: Final: '{ext.final_answer}' (Raw: '{ext.raw_answer[:60]}...')")

# ------------------------------------------------------------------------------
# 1. Visualize Stage 6 Evaluated Patches Overlaid Across All N Candidate Images
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6.2 * len(candidate_images), 6.2))
if len(candidate_images) == 1:
    axes = [axes]

for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    draw = ImageDraw.Draw(vis_img)
    img_records = [c for c in crop_records if c.get("img_idx") == img_idx]
    evid_count = sum(1 for c in img_records if c.get("is_evidence"))
    
    for c in img_records:
        is_ev = c.get("is_evidence", False)
        c_box = [int(round(float(v))) for v in c.get("box", [0, 0, 0, 0])]
        exp_box = [int(round(float(v))) for v in c.get("expanded_box", [0, 0, 0, 0])]
        
        if is_ev:
            draw.rectangle(c_box, outline="#00e676", width=4)
            draw.rectangle(exp_box, outline="#00e5ff", width=2)
        else:
            draw.rectangle(c_box, outline="#ff9800", width=2)
            
    thumb = vis_img.copy()
    thumb.thumbnail((768, 768))
    ax.imshow(thumb)
    is_t = (img_idx == target_pool_idx)
    color = "#10b981" if is_t else "#3b82f6"
    label = f"Image {img_idx} [{'TARGET' if is_t else 'DISTRACTOR'}]\n{evid_count} Evidence / {len(img_records)} Evaluated Crops"
    ax.set_title(label, fontsize=11, fontweight="bold", color=color)
    ax.axis("off")

plt.suptitle(f"Stage 6 Evaluated Patches Overlaid on Candidate Pool C (Total Crops: {len(crop_records)} | Green: Evidence | Orange: Filtered)", 
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. Visualize All Stage 6 Foveated Reading Crops & Predicted Text
# ------------------------------------------------------------------------------
n_read = len(crop_records)
if n_read > 0:
    ncols = min(4, n_read)
    nrows = (n_read + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 5.0 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    fig.suptitle(f"Stage 6: All Foveated Reading Crops & Predicted Text ({n_read} Total Crops)\nQuestion: '{real_question}'", 
                 fontsize=12.5, fontweight='bold', y=1.00)
    
    for i in range(n_read):
        ax = axes[i]
        c_rec = crop_records[i]
        img_idx = c_rec.get('img_idx', 0)
        exp_box = c_rec.get('expanded_box', [0, 0, 100, 100])
        loc_box = c_rec.get('local_target_box', None)
        is_ev = c_rec.get('is_evidence', False)
        ans = c_rec.get('raw_answer', '')
        src = c_rec.get('source_branch', '')
        is_t = (img_idx == target_pool_idx)
        
        crop_pil = candidate_images[img_idx].crop(tuple(exp_box))
        ax.imshow(crop_pil)
        
        if loc_box and len(loc_box) == 4:
            rect = mpatches.Rectangle((loc_box[0], loc_box[1]), loc_box[2] - loc_box[0], loc_box[3] - loc_box[1],
                                      linewidth=2.5, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            
        wrapped_ans = textwrap.fill(ans, width=32)
        status_tag = "EVIDENCE" if is_ev else "FILTERED"
        border_col = '#1b5e20' if is_ev else '#757575'
        tag = "TARGET" if is_t else "DISTRACTOR"
        
        ax.set_title(f"Crop {i} [Img {img_idx} - {tag}] | [{status_tag}]", fontsize=9.0,
                     color=border_col, fontweight='bold', pad=6)
        ax.set_xlabel(f'Predicted Text:\n"{wrapped_ans}"', fontsize=8.5,
                      fontweight='bold', color=border_col, labelpad=8)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col)
            spine.set_linewidth(2.5 if is_ev else 1.2)
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_read, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# CELL 10: END-TO-END PIPELINE EXECUTION & FULL TELEMETRY VERIFICATION
# ==============================================================================
print("=" * 75)
print(f"EXECUTING END-TO-END MULTI-SOURCE MSSearch PIPELINE (N = {len(candidate_images)} Images)")
print("=" * 75)

e2e_t0 = time.perf_counter()
e2e_response = get_mssearch_response(
    sam_model=sam3_engine,
    zoom_model=qwen_engine,
    siglip_model=siglip_engine,
    nlp_model=nlp_parser,
    annotation=target_sample,
    candidate_images=candidate_images,
    config=config_track_a
)
e2e_latency = time.perf_counter() - e2e_t0

telemetry = e2e_response.get("telemetry", {})
timings = telemetry.get("timings_sec", {})
primary_answer = e2e_response.get('primary_answer', '')
pred_letter = extract_multiple_choice_letter(primary_answer, parsed_options_dict)
is_e2e_correct = (pred_letter == gt_letter) if pred_letter and gt_letter else False

print(f"\n[End-to-End Execution Outcome]")
print(f"  - Primary Consolidated Answer     : '{primary_answer}'")
print(f"  - Predicted Option Letter         : '{pred_letter}' vs Ground Truth: '{gt_letter}' -> {'[CORRECT]' if is_e2e_correct else '[INCORRECT]'}")
print(f"  - Structured Answers List         : {e2e_response.get('answers')}")
print(f"  - Route Selected                  : {telemetry.get('route')}")
print(f"  - Evidence Images                 : {e2e_response.get('evidence_images')} (Target Pool Index: {target_pool_idx})")
print(f"  - Total Evaluated Patch BBoxes    : {len(e2e_response.get('stage6_patch_boxes', []))}")
print(f"  - Total End-to-End Latency        : {e2e_latency:.2f}s")

# Stage timing breakdown table
print(f"\n[Latency Profiling Breakdown]")
headers = ["Pipeline Stage", "Device / Models", "Latency (s)", "Share (%)"]
table_data = []

stage_meta = [
    ("phase1_direct_answer", "Phase 1: Direct Global Answer", "GPU 0 (Qwen2.5-VL)"),
    ("stage1_parser", "Stage 1: Question Parsing", "GPU 0 (Qwen2.5-VL)"),
    ("stage2_fast_path", "Stage 2: Tool Fast Path & Feat Cache", "GPU 1 (SAM3) + GPU 0 (Qwen2.5-VL)"),
    ("stage3_partitioning", "Stage 3: Saliency-Plateau Partitioning", "CPU + GPU 0 (SigLIP)"),
    ("stage4_grouping_gating", "Stage 4: SAM3 Feature Grouping & Gating", "GPU 1 (SAM3) + GPU 0 (Qwen2.5-VL)"),
    ("stage5_screening", "Stage 5: Continuous Enclosure Aggregation", "CPU Direct"),
    ("stage6_reading", "Stage 6: Foveated Reading & Multi-Answer", "GPU 0 (Qwen2.5-VL)")
]

total_recorded = sum(timings.values()) if timings else e2e_latency

for key, name, dev in stage_meta:
    lat = timings.get(key, 0.0)
    pct = (lat / total_recorded * 100.0) if total_recorded > 0 else 0.0
    table_data.append([name, dev, f"{lat:.2f}s", f"{pct:.1f}%"])

try:
    from tabulate import tabulate
    print(tabulate(table_data, headers=headers, tablefmt="github"))
except ImportError:
    for row in [headers] + table_data:
        print(f"  {row[0]:<45} | {row[1]:<30} | {row[2]:<10} | {row[3]:<10}")

# ------------------------------------------------------------------------------
# Funnel Visualization: Candidate Reduction Across Pipeline Stages
# ------------------------------------------------------------------------------
funnel_stages = ["Input Images (N)", "Stage 3 Patches", "Stage 4 Retained", "Stage 5 SubCanvases", "Stage 6 Evaluated", "Final Answer"]
funnel_counts = [
    len(candidate_images),
    len(bounded_patches),
    len(remaining_patches),
    len(unified_pool),
    len(crop_records) if crop_records else len(unified_pool),
    1
]

fig, ax = plt.subplots(figsize=(9, 4))
y_pos = np.arange(len(funnel_stages))
colors = ["#3b82f6", "#0284c7", "#0d9488", "#10b981", "#8b5cf6", "#ec4899"]

bars = ax.barh(y_pos, funnel_counts, color=colors, height=0.55)
ax.set_yticks(y_pos)
ax.set_yticklabels(funnel_stages, fontsize=10, fontweight="bold")
ax.invert_yaxis()  # top-down funnel
ax.set_xlabel("Count / Volume", fontsize=10, fontweight="bold")
ax.set_title("MSSearch Candidate Pruning & Reduction Funnel Across Stages", fontsize=12, fontweight="bold")

for bar, count in zip(bars, funnel_counts):
    width = bar.get_width()
    ax.text(width + max(funnel_counts)*0.015, bar.get_y() + bar.get_height()/2,
            f"{count}", va='center', ha='left', fontsize=10, fontweight='bold', color='#1e293b')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("=" * 75)
print(f"DIAGNOSTIC BENCHMARK COMPLETE FOR N = {len(candidate_images)} CANDIDATE IMAGES!")
print("=" * 75)
